## Semantic Similarity

In [1]:
import pandas as pd
from scipy.stats import pearsonr
from similarity_scores import noun_verb_wup

def wup_correlation(file_path, name):
    df = pd.read_csv(file_path)
    
    if "wup" not in df.columns:
        df["wup"] = df.apply(lambda row: noun_verb_wup(row["S1"], row["S2"]), axis=1)
        df.to_csv(file_path, index=False)

    r, _ = pearsonr(df["wup"], df["score"])
    print(f"{name:<12}Pearson correlation: {r:.4f}")


wup_correlation("data/stss131.csv", "STSS-131")
wup_correlation("data/sts12.csv", "STS-2012")
wup_correlation("data/sts13.csv", "STS-2013")
wup_correlation("data/sts14.csv", "STS-2014")
wup_correlation("data/sts15.csv", "STS-2015")
wup_correlation("data/sts16.csv", "STS-2016")
wup_correlation("data/stsb.csv", "STSB")
wup_correlation("data/sickr.csv", "SICK-R")
wup_correlation("data/mrpc.csv", "MRPC")

STSS-131    Pearson correlation: 0.3824
STS-2012    Pearson correlation: 0.1751
STS-2013    Pearson correlation: 0.2818
STS-2014    Pearson correlation: 0.3136
STS-2015    Pearson correlation: 0.3570
STS-2016    Pearson correlation: 0.2093
STSB        Pearson correlation: 0.2428
SICK-R      Pearson correlation: 0.4445
MRPC        Pearson correlation: 0.1759


In [2]:
import gensim.downloader as api

w2v_model = api.load("word2vec-google-news-300")

In [3]:
from similarity_scores import sentence_similarity

def w2v_correlation(file_path, name):
    df = pd.read_csv(file_path)
    
    if "w2v" not in df.columns:
        df["w2v"] = df.apply(lambda row: sentence_similarity(w2v_model, row["S1"], row["S2"]), axis=1)
        df.to_csv(file_path, index=False)

    r, _ = pearsonr(df["w2v"], df["score"])
    print(f"{name:<12}Pearson correlation: {r:.4f}")
    
    
w2v_correlation("data/stss131.csv", "STSS-131")
w2v_correlation("data/sts12.csv", "STS-2012")
w2v_correlation("data/sts13.csv", "STS-2013")
w2v_correlation("data/sts14.csv", "STS-2014")
w2v_correlation("data/sts15.csv", "STS-2015")
w2v_correlation("data/sts16.csv", "STS-2016")
w2v_correlation("data/stsb.csv", "STSB")
w2v_correlation("data/sickr.csv", "SICK-R")
w2v_correlation("data/mrpc.csv", "MRPC")

STSS-131    Pearson correlation: 0.7588
STS-2012    Pearson correlation: 0.4317
STS-2013    Pearson correlation: 0.6711
STS-2014    Pearson correlation: 0.6380
STS-2015    Pearson correlation: 0.7002
STS-2016    Pearson correlation: 0.5860
STSB        Pearson correlation: 0.6274
SICK-R      Pearson correlation: 0.7006
MRPC        Pearson correlation: 0.3968


In [4]:
from sentence_transformers import SentenceTransformer

ft_model = api.load("fasttext-wiki-news-subwords-300")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
roberta_model = SentenceTransformer("stsb-roberta-base-v2")
albert_model = SentenceTransformer("paraphrase-albert-small-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

In [5]:
from similarity_scores import bert_similarity

def evaluate_similarity_models(file_path, file_name):
    df = pd.read_csv(file_path)

    similarity_models = {
        "wup": lambda s1, s2: noun_verb_wup(s1, s2),
        "w2v": lambda s1, s2: sentence_similarity(w2v_model, s1, s2),
        "ft": lambda s1, s2: sentence_similarity(ft_model, s1, s2),
        "sbert": lambda s1, s2: bert_similarity(sbert_model, s1, s2),
        "roberta": lambda s1, s2: bert_similarity(roberta_model, s1, s2),
        "albert": lambda s1, s2: bert_similarity(albert_model, s1, s2),
    }

    for name, fn in similarity_models.items():
        if name not in df.columns:
            df[name] = df.apply(lambda row: fn(row["S1"], row["S2"]), axis=1)

    df.to_csv(file_path, index=False)
    
    wup_r, _ = pearsonr(df["wup"], df["score"])
    w2v_r, _ = pearsonr(df["w2v"], df["score"])
    ft_r, _ = pearsonr(df["ft"], df["score"])
    sbert_r, _ = pearsonr(df["sbert"], df["score"])
    roberta_r, _ = pearsonr(df["roberta"], df["score"])
    albert_r, _ = pearsonr(df["albert"], df["score"])
    
    print(
        f"{file_name:<12}Wu & Palmer: {wup_r:.4f} | Word2vec: {w2v_r:.4f}" +
        f" | FastText: {ft_r:.4f} | SBERT: {sbert_r:.4f}" + 
        f" | RoBERTa: {roberta_r:.4f} | ALBERT: {albert_r:.4f}"
    )
  

print("Pearson correlation coefficients")
evaluate_similarity_models("data/stss131.csv", "STSS-131")
evaluate_similarity_models("data/sts12.csv", "STS-2012")
evaluate_similarity_models("data/sts13.csv", "STS-2013")
evaluate_similarity_models("data/sts14.csv", "STS-2014")
evaluate_similarity_models("data/sts15.csv", "STS-2015")
evaluate_similarity_models("data/sts16.csv", "STS-2016")
evaluate_similarity_models("data/stsb.csv", "STSB")
evaluate_similarity_models("data/sickr.csv", "SICK-R")
evaluate_similarity_models("data/mrpc.csv", "MRPC")

Pearson correlation coefficients
STSS-131    Wu & Palmer: 0.3824 | Word2vec: 0.7588 | FastText: 0.4914 | SBERT: 0.9180 | RoBERTa: 0.9048 | ALBERT: 0.9054
STS-2012    Wu & Palmer: 0.1751 | Word2vec: 0.4317 | FastText: 0.2867 | SBERT: 0.6163 | RoBERTa: 0.7867 | ALBERT: 0.6236
STS-2013    Wu & Palmer: 0.2818 | Word2vec: 0.6711 | FastText: 0.2855 | SBERT: 0.8015 | RoBERTa: 0.9067 | ALBERT: 0.8115
STS-2014    Wu & Palmer: 0.3136 | Word2vec: 0.6380 | FastText: 0.2000 | SBERT: 0.8080 | RoBERTa: 0.9061 | ALBERT: 0.8074
STS-2015    Wu & Palmer: 0.3570 | Word2vec: 0.7002 | FastText: 0.3592 | SBERT: 0.8474 | RoBERTa: 0.8988 | ALBERT: 0.8406
STS-2016    Wu & Palmer: 0.2093 | Word2vec: 0.5860 | FastText: 0.2707 | SBERT: 0.7794 | RoBERTa: 0.8367 | ALBERT: 0.7956
STSB        Wu & Palmer: 0.2428 | Word2vec: 0.6274 | FastText: 0.2739 | SBERT: 0.8407 | RoBERTa: 0.9438 | ALBERT: 0.8448
SICK-R      Wu & Palmer: 0.4445 | Word2vec: 0.7006 | FastText: 0.4196 | SBERT: 0.8392 | RoBERTa: 0.8438 | ALBERT: 0.8088

In [6]:
from util import lemma_filter, stopword_filter, stop_lemma_filter, alphabet_filter

def evaluate_filtered_similarity(file_path, filter_method):
    df = pd.read_csv(file_path)
    
    similarity_models = {
        f"{filter_method}_wup": lambda s1, s2: noun_verb_wup(s1, s2),
        f"{filter_method}_w2v": lambda s1, s2: sentence_similarity(w2v_model, s1, s2),
        f"{filter_method}_ft": lambda s1, s2: sentence_similarity(ft_model, s1, s2),
        f"{filter_method}_sbert": lambda s1, s2: bert_similarity(sbert_model, s1, s2),
        f"{filter_method}_roberta": lambda s1, s2: bert_similarity(roberta_model, s1, s2),
        f"{filter_method}_albert": lambda s1, s2: bert_similarity(albert_model, s1, s2),
    }
    filters = {
        "lemma": lambda sentence: lemma_filter(sentence),
        "stop": lambda sentence: stopword_filter(sentence),
        "stop_lemma": lambda sentence: stop_lemma_filter(sentence),
        "alphabet": lambda sentence: alphabet_filter(sentence)
    }
        
    for name, fn in similarity_models.items():
        if name not in df.columns:
            s1 = [filters[filter_method](sentence) for sentence in df["S1"]]
            s2 = [filters[filter_method](sentence) for sentence in df["S2"]]
            df[name] = [fn(a, b) for a, b in zip(s1, s2)]

    df.to_csv(file_path, index=False)
    
    
for filter_method in ["lemma", "stop", "stop_lemma", "alphabet"]:
    evaluate_filtered_similarity("data/stss131.csv", filter_method)
    evaluate_filtered_similarity("data/sts12.csv", filter_method)
    evaluate_filtered_similarity("data/sts13.csv", filter_method)
    evaluate_filtered_similarity("data/sts14.csv", filter_method)
    evaluate_filtered_similarity("data/sts15.csv", filter_method)
    evaluate_filtered_similarity("data/sts16.csv", filter_method)
    evaluate_filtered_similarity("data/stsb.csv", filter_method)
    evaluate_filtered_similarity("data/sickr.csv", filter_method)
    evaluate_filtered_similarity("data/mrpc.csv", filter_method)

In [7]:
import numpy as np
from collections import defaultdict

file_names = [
    "data/stss131.csv", "data/sts12.csv","data/sts13.csv","data/sts14.csv",
    "data/sts15.csv","data/sts16.csv","data/stsb.csv","data/sickr.csv","data/mrpc.csv"
]
scores = defaultdict(list)

for name in file_names:
    df = pd.read_csv(name)
    for i in range(3, 33):
        r, _ = pearsonr(df.iloc[:, i], df["score"])
        scores[df.columns[i]].append(r)

methods = {
    "Wu & Palmer": "wup", "Word2Vec": "w2v", "FastText": "ft",
    "SBERT": "sbert", "RoBERTa": "roberta", "ALBERT": "albert",
}
variants = {
    "Default": "", "Lemmatization": "lemma_", "Stopword": "stop_",
    "Lemma & stop": "stop_lemma_", "Alphabet only": "alphabet_",
}
means = {key: np.mean(values) for key, values in scores.items()}

table = pd.DataFrame({
    variant: {method: means[prefix + key] for method, key in methods.items()}
    for variant, prefix in variants.items()
})

print(table.round(4).to_string())


             Default  Lemmatization  Stopword  Lemma & stop  Alphabet only
Wu & Palmer   0.2869         0.2702    0.3060        0.2938         0.2930
Word2Vec      0.6123         0.6186    0.6326        0.6364         0.6193
FastText      0.3181         0.3165    0.4202        0.3889         0.3798
SBERT         0.7613         0.7547    0.7316        0.7295         0.7484
RoBERTa       0.8402         0.8259    0.7810        0.7787         0.8081
ALBERT        0.7627         0.7581    0.7322        0.7277         0.7511


In [ ]:
from itertools import islice
from util import get_verb_adjective, get_synonyms
from similarity_scores import sentence_similarity

def ranked_synonyms(word, model):
    """Return (token, sorted_synonyms), ranked by a similarity score, or None."""
    synonyms = get_synonyms(word)

    if synonyms:
        return word[0], sorted(
            synonyms, 
            key=lambda synonym: sentence_similarity(model, word[0], synonym),
            reverse=True
        )
    return None

def candidates(sample):
    for row in sample.itertuples(index=False):
        p1 = get_verb_adjective(row.S1)
        p2 = get_verb_adjective(row.S2)
        
        if not p1 or not p2: continue
        
        synonyms = [ranked_synonyms(w, w2v_model) for p in (p1, p2) for w in p]
        
        if any(result is None for result in synonyms): continue
        
        yield row, synonyms  

def synonym_candidates(file_name):
    df = pd.read_csv(file_name)
    sample = df[["S1", "S2", "score"]].sample(frac=1)
        
    for row, (v1, a1, v2, a2) in islice(candidates(sample), 4):
        print(f'"{row.S1}","{row.S2}",{row.score}')
        print(f"{v1[0]} - {v1[1]}\n{a1[0]} - {a1[1]}")
        print(f"{v2[0]} - {v2[1]}\n{a2[0]} - {a2[1]}\n")
        

synonym_candidates(file_names[0])

In [11]:
print("Pearson correlation coefficients")
evaluate_similarity_models("data/synonyms.csv", "Synonyms")

Pearson correlation coefficients
Synonyms    Wu & Palmer: 0.1300 | Word2vec: 0.3840 | FastText: 0.2281 | SBERT: 0.5145 | RoBERTa: 0.6503 | ALBERT: 0.6291
